### The minima are in $x_1 = 1.75$ and $x_2 = 4$

In [93]:
######## I HAVE INDICATED WITH !!! THE PARTS WHICH HAVE BEEN ADAPTED FOR THIS PROBLEM


################### IMPORT THE REQUIRED MODULES # pip install dwave-neal

import numpy as np
import neal
import math
import itertools
import sympy as sp
import time

In [94]:
################## CHOOSE THE DWAVE SIMULATED ANNEALING SAMPLER

sampler = neal.SimulatedAnnealingSampler()

In [95]:
################## MAP SPINS {-1,1} TO QUBITS {0,1}

def taufromsig(sig):
    return (sig+1)/2


################### !!! CONTINUOUS BINARY ENCODING. fractional_part IS DEFINED LATER

def Xfromtau(tau): 
    num=0
    for l in range(beta):
        num += tau[l]*2**(l-fractional_part)
    return num

In [96]:
################## THIS IS TO AUTOMATISE THE REDUCTION OPTIMISING THE NUMBER OF AUXILIARY SPINS. 

################## YOU DO NOT WANT TO TOUCH THIS BOX


def count_couplings(*clists):
    
    occ_list = np.zeros(shape=(0,3))
    
    for l in range(len(clists)):
        
        ncol = clists[l].shape[1]
    
        for i in range(len(clists[l])):
            
            
            minus1 = np.where(clists[l][i][0:ncol-1] == -1)[0]
        
            spins = []
            
            
            if len(minus1) == 0: 
                
                spins = clists[l][i][0:ncol-1]
            
            elif minus1[0] == 2: continue 
        
            else:
                
                
                for j in range(minus1[0]): spins.append(clists[l][i][j])
                    
            
            pairs = list(itertools.combinations(spins, 2)) 
            
            
            for j in range(len(pairs)):
        
                occ = np.where((occ_list[:,0] == pairs[j][0]) & (occ_list[:,1] == pairs[j][1]))[0]

                if len(occ) == 1: occ_list[occ[0]][2] += 1
            
                else: occ_list = np.append(occ_list, np.array([[pairs[j][0], pairs[j][1], 1]]),axis=0)
                    
        
    return occ_list

def replace_pairs(num_spin,most,*higher_coup):
    
    for l in range(len(higher_coup)):
    
        for i in range(len(higher_coup[l])):
            
            ncol = higher_coup[l].shape[1]
            
            minus1 = np.where(higher_coup[l][i][0:ncol-1] == -1)[0]
        
            inds = []
        
            if len(minus1) == 0: 
                
                p_minus1 = ncol-2
                for q in range(ncol-1): inds.append(q)
                

            
            else:
                
                p_minus1 = minus1[0]-1
                for j in range(minus1[0]): inds.append(j)
                    
                
               
            pairs = list(itertools.combinations(inds, 2)) 
            
            
            for j in range(len(pairs)):
                
                i1 = int(pairs[j][0])
                i2 = int(pairs[j][1])
                
    

                if higher_coup[l][i][i1] == most[0] and higher_coup[l][i][i2] == most[1]:
                    
                    higher_coup[l][i][i1] = num_spin
                    higher_coup[l][i][i2] = higher_coup[l][i][p_minus1]
                    higher_coup[l][i][p_minus1] = -1
                    
                    higher_coup[l][i][0:p_minus1].sort()
                    
                    
    return higher_coup

def make_quadratic(num_spin,L,quadratic,linear,*higher_coup):
    
    aux_spin = num_spin-1
    
    while True:
        
        occ_list = count_couplings(*higher_coup)
        
        if len(occ_list) == 0: break
            
        occ_list = occ_list[np.argsort(occ_list[:,2])]
        most = np.array([occ_list[len(occ_list)-1][0],occ_list[len(occ_list)-1][1]])
    
        print("{",int(most[0]),",",int(most[1]),"}")
        aux_spin += 1 
        
        quadratic = np.append(quadratic, np.array([[aux_spin, most[0],-2*L]]),axis=0)
        quadratic = np.append(quadratic, np.array([[aux_spin, most[1],-2*L]]),axis=0)
        linear = np.append(linear, np.array([[aux_spin,3*L]]),axis=0)
            
        occ = np.where((quadratic[:,0] == most[0]) & (quadratic[:,1] == most[1]))[0]
            
        if len(occ) > 0: 
            linear[len(linear)-1][1] += quadratic[occ[0]][2]
            quadratic[occ[0]][2] = L
        
        else:
            quadratic = np.append(quadratic, np.array([[most[0], most[1],L]]),axis=0)
        
        higher_coup = replace_pairs(aux_spin,most,*higher_coup)
    
        occ_list = np.zeros(shape=(0,3))
        
    
    for l in range(len(higher_coup)):
        
        ncol = higher_coup[l].shape[1]
        
        for i in range(len(higher_coup[l])):
            
            if higher_coup[l][i][1] == -1:
                
                occ = np.where(linear[:,0] == higher_coup[l][i][0])[0]
                
                if len(occ) == 0:
                    linear = np.append(linear, np.array([[higher_coup[l][i][0], higher_coup[l][i][ncol-1]]]),axis=0)
                    
                else:
                    linear[occ[0]][1] += higher_coup[l][i][ncol-1]
            else:    
                
                occ = np.where((quadratic[:,0] == higher_coup[l][i][0]) & (quadratic[:,1] == higher_coup[l][i][1]))[0]
                occ1 = np.where((quadratic[:,1] == higher_coup[l][i][0]) & (quadratic[:,0] == higher_coup[l][i][1]))[0]
                
                if len(occ) == 0 and len(occ1) == 0:
                    quadratic = np.append(quadratic, np.array([[higher_coup[l][i][0],higher_coup[l][i][1], higher_coup[l][i][ncol-1]]]),axis=0)
    
                elif len(occ) != 0:
                    quadratic[occ[0]][2] += higher_coup[l][i][ncol-1]
                
                elif len(occ1) != 0:
                    quadratic[occ1[0]][2] += higher_coup[l][i][ncol-1]
    
    print("Number of auxiliary spins: ",aux_spin + 1 - n_var*beta)
    
    
    return quadratic, linear, aux_spin 

In [97]:
################## !!! YOU DO NOT NEED TO TEST ANY SOLUTION. YOU DO NOT KNOW WHERE THE MINIMA ARE

#def itisasolution(lambdas):
    
#    z = lambdas[0:n_var]

#    return  (z[0] + z[1] - 6)**2

In [98]:
########### !!! 

integer_part = 3
fractional_part = 5
beta = integer_part + fractional_part

###########


n_var = 1     ################## !!!

shift = 0     ################## EVENTUAL SHIFT IN THE BINARY REPRESENTATION

max_exp = 4   ################## !!! THE LARGEST EXPONENT THAT APPEARS IN THE HAMILTONIAN

In [99]:
################## EQS IS A LIST CONTAINING THE EQUATION(S) WE WANT TO SOLVE IN TERM OF THE SYMPY SYMBOLS Z

eqs = []

z = sp.symarray("z",n_var)

eq = z[0]**4 - 23/2*z[0]**3 + 753/16*z[0]**2 - 161/2*z[0] #### !!!
eqs.append(eq) 

In [100]:
################## THIS IS JUST TO CHOOSE, IF NEEDED, A SUBSET OF THE EQUATIONS/VARIABLES INVOLVED.

################## YOU CAN INGORE IT

eq_chosen = []
var_chosen = []

n_eq = len(eqs)

var_chosen= [n for n in range(n_var)]
eq_chosen = [n for n in range(n_eq)]

In [101]:
################## !!! CONTINOUS BINARY ENCODING

sym = sp.symarray("x", n_var*beta)                       
sigmas = np.array([2**n for n in range(-fractional_part, integer_part)])

In [102]:
################## variables IS A LIST CONTAINING THE BINARY REPRESENTATION OF THE VARIABLES INVOLVED

variables = []



for i in range(n_var): 
    variables.append(np.tensordot(sigmas, sym[i*beta:(i+1)*beta], axes=(0,0)) + shift) 
    
print(variables)

[0.03125*x_0 + 0.0625*x_1 + 0.125*x_2 + 0.25*x_3 + 0.5*x_4 + 1.0*x_5 + 2.0*x_6 + 4.0*x_7]


In [103]:
################## CONSTRUCT THE HAMILTONIAN AS A SYMPY POLYNOMIAL 


sq = 0
for i in range(n_eq): sq += eqs[eq_chosen[i]]
print("The Hamiltonian is: \n\n",sq)
squares = sq.copy()


for i in range(n_var): sq = sq.subs(z[i],variables[i])  ############ IN THE HAMILTONIAN, WE SUBSTITUTE Z WITH ITS BINARY REPRESENTATION
    
start = time.time()
H = sp.poly((sq),*sym)
print("\n\nHamiltonian written in terms of taus:", time.time()-start)

The Hamiltonian is: 

 z_0**4 - 11.5*z_0**3 + 47.0625*z_0**2 - 80.5*z_0


Hamiltonian written in terms of taus: 0.07703900337219238


In [104]:
##################### REPLACE TAU^N WITH TAU AS TAU = {0,1}

print("\nReplace powers...")



lsym = [n for n in range(n_var*beta)]
mapping = {sym[i]**j: sym[i] for i,j in list(itertools.product(lsym, [n for n in range(max_exp+1)]))}

start = time.time()
H = H.xreplace(mapping)
print("Done in", time.time()-start)
        
H = sp.poly(H,*sym)

print(H)


Replace powers...
Done in 0.11339378356933594
Poly(0.00146484375*x_0*x_1*x_2*x_3 + 0.0029296875*x_0*x_1*x_2*x_4 + 0.005859375*x_0*x_1*x_2*x_5 + 0.01171875*x_0*x_1*x_2*x_6 + 0.0234375*x_0*x_1*x_2*x_7 - 0.016204833984375*x_0*x_1*x_2 + 0.005859375*x_0*x_1*x_3*x_4 + 0.01171875*x_0*x_1*x_3*x_5 + 0.0234375*x_0*x_1*x_3*x_6 + 0.046875*x_0*x_1*x_3*x_7 - 0.03167724609375*x_0*x_1*x_3 + 0.0234375*x_0*x_1*x_4*x_5 + 0.046875*x_0*x_1*x_4*x_6 + 0.09375*x_0*x_1*x_4*x_7 - 0.0604248046875*x_0*x_1*x_4 + 0.09375*x_0*x_1*x_5*x_6 + 0.1875*x_0*x_1*x_5*x_7 - 0.109130859375*x_0*x_1*x_5 + 0.375*x_0*x_1*x_6*x_7 - 0.17138671875*x_0*x_1*x_6 - 0.1552734375*x_0*x_1*x_7 + 0.177581787109375*x_0*x_1 + 0.01171875*x_0*x_2*x_3*x_4 + 0.0234375*x_0*x_2*x_3*x_5 + 0.046875*x_0*x_2*x_3*x_6 + 0.09375*x_0*x_2*x_3*x_7 - 0.0626220703125*x_0*x_2*x_3 + 0.046875*x_0*x_2*x_4*x_5 + 0.09375*x_0*x_2*x_4*x_6 + 0.1875*x_0*x_2*x_4*x_7 - 0.119384765625*x_0*x_2*x_4 + 0.1875*x_0*x_2*x_5*x_6 + 0.375*x_0*x_2*x_5*x_7 - 0.21533203125*x_0*x_2*x_5 +

In [105]:
################## EXTRACT THE COUPLINGS AS A DICTIONARY, YOU DO NOT NEED TO EDIT THIS

dict_coup = {}

coup_list = list(H.as_dict().items())

for i in range(len(coup_list)):
    
    spins = np.where(np.array(coup_list[i][0][:]) == 1)[0]
    
    if len(spins) > 0:
        
        spins = np.append(spins,coup_list[i][1])
        
        if len(spins)-1 in dict_coup:
            
            dict_coup[len(spins)-1] = np.append(dict_coup[len(spins)-1],np.array([spins]),axis=0)
             
        else:
            
            dict_coup[len(spins)-1] = np.zeros(shape = (0,len(spins)))
            dict_coup[len(spins)-1] = np.append(dict_coup[len(spins)-1],np.array([spins]),axis=0)
            
quad = dict_coup[2]
lin = dict_coup[1]



del dict_coup[1]
del dict_coup[2]


In [106]:
################## AUTOMATIC REDUCTION OF THE HAMILTONIAN

L = 2 * int(max(list(H.as_dict().values())))

reduced = make_quadratic(n_var*beta,L,quad,lin,*list(dict_coup.values()))

{ 0 , 1 }
{ 5 , 6 }
{ 4 , 7 }
{ 2 , 3 }
{ 0 , 2 }
{ 3 , 6 }
{ 3 , 5 }
{ 1 , 2 }
{ 6 , 7 }
{ 5 , 7 }
{ 4 , 6 }
{ 4 , 5 }
{ 0 , 7 }
{ 1 , 7 }
{ 1 , 4 }
{ 0 , 4 }
{ 2 , 8 }
{ 3 , 9 }
{ 1 , 10 }
{ 0 , 10 }
{ 1 , 11 }
{ 2 , 9 }
{ 3 , 8 }
{ 0 , 11 }
{ 2 , 10 }
Number of auxiliary spins:  25


In [107]:
################## CONSTRUCT h AND J

quad = reduced[0].copy()
lin = reduced[1].copy()
aux_spin = reduced[2]

J = np.zeros(shape=(aux_spin+1,aux_spin+1))
h = np.zeros(shape=(aux_spin+1))
   
    
for i in range(len(quad)):
    
    qi0 = int(quad[i][0])
    qi1 = int(quad[i][1])
    qi2 = quad[i][2]
        
    J[qi0][qi1] = qi2/4
    
    
        
    h[qi0] += qi2/4
    h[qi1] += qi2/4
        
for i in range(len(lin)):
    
    li0 = int(lin[i][0])
    li1 = lin[i][1]
        
    h[li0] += li1/2

In [108]:
nreads = 2000           # NUMBER OF ANNEAL RUNS     

answer = sampler.sample_ising(h, J, num_reads=nreads)         # SIMULATED ANNEALING

print(answer)           # PRINT THE OUTPUT


      0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 ... 32       energy num_oc.
32   -1 -1 -1 -1 -1 -1 -1 +1 -1 -1 -1 -1 -1 -1 -1 ... -1  -4756.67183       1
59   -1 -1 -1 +1 +1 +1 -1 -1 -1 -1 -1 -1 -1 -1 +1 ... -1  -4756.67183       1
95   -1 -1 -1 +1 +1 +1 -1 -1 -1 -1 -1 -1 -1 -1 +1 ... -1  -4756.67183       1
154  -1 -1 -1 +1 +1 +1 -1 -1 -1 -1 -1 -1 -1 -1 +1 ... -1  -4756.67183       1
156  -1 -1 -1 -1 -1 -1 -1 +1 -1 -1 -1 -1 -1 -1 -1 ... -1  -4756.67183       1
189  -1 -1 -1 -1 -1 -1 -1 +1 -1 -1 -1 -1 -1 -1 -1 ... -1  -4756.67183       1
277  -1 -1 -1 +1 +1 +1 -1 -1 -1 -1 -1 -1 -1 -1 +1 ... -1  -4756.67183       1
279  -1 -1 -1 -1 -1 -1 -1 +1 -1 -1 -1 -1 -1 -1 -1 ... -1  -4756.67183       1
317  -1 -1 -1 -1 -1 -1 -1 +1 -1 -1 -1 -1 -1 -1 -1 ... -1  -4756.67183       1
374  -1 -1 -1 -1 -1 -1 -1 +1 -1 -1 -1 -1 -1 -1 -1 ... -1  -4756.67183       1
471  -1 -1 -1 -1 -1 -1 -1 +1 -1 -1 -1 -1 -1 -1 -1 ... -1  -4756.67183       1
576  -1 -1 -1 -1 -1 -1 -1 +1 -1 -1 -1 -1 -1 -1 -1 ... -1  -4756.

In [109]:
################### THIS IS JUST TO CONVERT FROM SPIN VALUES TO NUMBERS


Lambdasig = []
Lambda = []

for i in range(n_var): 
    
    Lambdasig.append(np.zeros(shape=(nreads,beta)))
    Lambda.append(np.zeros(shape=(nreads)))

values = np.zeros(shape=(nreads))

for i in range(nreads):
    for j in range(beta):
        for k in range(n_var):
        
            Lambdasig[k][i][j] = answer.record[i][0][j+k*beta]
        
    for k in range(n_var):

        Lambda[k][i] = Xfromtau(taufromsig(Lambdasig[k][i])) + shift
        
    current_Lambda = list(zip(*Lambda))[i]
    
    #values[i] = itisasolution(current_Lambda) ### !!! YOU DO NOT NEED TO TEST ANYTHING
    
    
    #if values[i] == 0:  # !!! YOU DO NOT NEED TO TEST ANYTHING
    
    if answer.record[i][1] == answer.first.energy: # AND IF IT IS ACTUALLY THE MINIMUM STATE FOUND
        print("\n\n\nSolution:",i,current_Lambda) # THEN ---> PRINT THE NUMBERS 
                
         




Solution: 32 (4.0,)



Solution: 59 (1.75,)



Solution: 95 (1.75,)



Solution: 154 (1.75,)



Solution: 156 (4.0,)



Solution: 189 (4.0,)



Solution: 277 (1.75,)



Solution: 279 (4.0,)



Solution: 317 (4.0,)



Solution: 374 (4.0,)



Solution: 471 (4.0,)



Solution: 576 (4.0,)



Solution: 620 (1.75,)



Solution: 625 (1.75,)



Solution: 648 (4.0,)



Solution: 661 (4.0,)



Solution: 717 (4.0,)



Solution: 732 (4.0,)



Solution: 768 (4.0,)



Solution: 794 (4.0,)



Solution: 815 (4.0,)



Solution: 870 (4.0,)



Solution: 911 (1.75,)



Solution: 929 (4.0,)



Solution: 981 (4.0,)



Solution: 988 (1.75,)



Solution: 1017 (1.75,)



Solution: 1060 (4.0,)



Solution: 1109 (4.0,)



Solution: 1137 (4.0,)



Solution: 1215 (4.0,)



Solution: 1247 (4.0,)



Solution: 1361 (4.0,)



Solution: 1392 (4.0,)



Solution: 1449 (4.0,)



Solution: 1468 (4.0,)



Solution: 1522 (1.75,)



Solution: 1556 (4.0,)



Solution: 1557 (4.0,)



Solution: 1590 (4.0,)



Solution: 1683 (